# Proyecto LSP — Pipeline PUCP-305 en Colab

Procesa el **dataset PUCP-305 glosas** (~305 señas LSP de PUCP) end-to-end:

1. Descarga el .zip (2.88 GB) desde el repositorio de PUCP.
2. Descomprime y descubre pares video+ELAN.
3. Explora los tiers ELAN y cuenta glosas para decidir vocabulario.
4. Segmenta videos por anotaciones → MediaPipe Holistic → tensor `(30, 258)`.
5. Filtra a glosas con suficientes muestras (≥ `MIN_MUESTRAS_POR_CLASE`).
6. Entrena modelo LSTM con augmentation, evalúa y guarda.

**Pre-requisitos:**
- GPU T4 activada.
- ~10 GB libres en Drive para `.npy` finales (los videos vivien solo en Colab temporal).

**Tiempo estimado:** 1.5-3 horas (la mayoría es descarga + MediaPipe).

## Setup — clonar repo, instalar y montar Drive

In [ ]:
%cd /content
!rm -rf /content/Proyecto_Percepcion
!git clone https://github.com/Jtarazona00/Proyecto_Percepcion.git /content/Proyecto_Percepcion
%cd /content/Proyecto_Percepcion

import os
os.environ['DATASET'] = 'pucp305'  # IMPORTANTE: activar el dataset antes de importar config

!pip install -q tensorflow==2.18.0 mediapipe==0.10.14

import tensorflow as tf
from google.colab import drive
drive.mount('/content/drive')
print('GPU disponible:', tf.config.list_physical_devices('GPU'))
print('TF version:', tf.__version__)

## Paso 1 — Descargar el .zip de PUCP

El archivo es ~2.88 GB. Si la celda se interrumpe, vuelve a ejecutarla (`wget -c` reanuda).

In [ ]:
from pathlib import Path

LOCAL_ZIP = Path('/content/pucp305.zip')
LOCAL_EXTRACTED = Path('/content/pucp305_data')
PROCESSED_DIR_DRIVE = Path('/content/drive/MyDrive/PUCP305_processed')
MODELS_DIR_DRIVE = Path('/content/drive/MyDrive/PUCP305_models')

for d in (PROCESSED_DIR_DRIVE, MODELS_DIR_DRIVE, LOCAL_EXTRACTED):
    d.mkdir(parents=True, exist_ok=True)

# Descarga con reanudacion (-c). HTTP 200 OK, no requiere auth.
!wget -c -q --show-progress -O {LOCAL_ZIP} 'https://datos.pucp.edu.pe/api/access/datafile/21028'

import os
size_gb = os.path.getsize(LOCAL_ZIP) / 1024**3
print(f'\nArchivo descargado: {size_gb:.2f} GB')
assert size_gb > 2.5, 'La descarga parece incompleta. Re-ejecuta esta celda.'

## Paso 2 — Descomprimir

Tarda 2-5 minutos. Verifica el resultado contando archivos.

In [ ]:
!unzip -n -q {LOCAL_ZIP} -d {LOCAL_EXTRACTED}

videos = list(LOCAL_EXTRACTED.rglob('*.mp4')) + list(LOCAL_EXTRACTED.rglob('*.mov'))
eafs = list(LOCAL_EXTRACTED.rglob('*.eaf'))
print(f'Videos encontrados: {len(videos)}')
print(f'Archivos ELAN (.eaf): {len(eafs)}')
if videos:
    print(f'\nEjemplos:')
    for v in videos[:3]:
        print(f'  video: {v.relative_to(LOCAL_EXTRACTED)}')
    for e in eafs[:3]:
        print(f'  eaf:   {e.relative_to(LOCAL_EXTRACTED)}')

## Paso 3 — Explorar estructura ELAN

Lista los nombres de tiers en algunos .eaf para identificar cuál contiene las glosas. Luego cuenta glosas únicas y muestras por glosa.

In [ ]:
import sys
sys.path.insert(0, '/content/Proyecto_Percepcion')

from src.preprocesamiento.parse_elan import listar_tiers, parsear_eaf
from collections import Counter

# Que tiers existen?
tiers_counter = Counter()
for eaf in eafs[:50]:  # muestra de 50
    for t in listar_tiers(eaf):
        tiers_counter[t] += 1

print('Tiers mas comunes (top 20):')
for t, n in tiers_counter.most_common(20):
    print(f'  {t!r}: aparece en {n} eafs')

In [ ]:
# Ajusta TIERS_INTERES con los nombres reales que veas arriba.
# Tipicamente: 'GLOSA', 'GLOSS_ES', 'ID_GLOSS', 'Glosa', 'TRANSLATION'...
# Si None, usa TODOS los tiers (mas glosas pero mas ruido).
TIERS_INTERES = None  # ej: ['GLOSA', 'ID_GLOSA']

from src.preprocesamiento.extraer_dataset_pucp305 import (
    encontrar_pares_video_eaf, descubrir_glosas
)

pares = encontrar_pares_video_eaf(LOCAL_EXTRACTED)
print(f'Pares (video, .eaf) emparejados: {len(pares)}')

conteo, glosas_validas = descubrir_glosas(pares, tiers=TIERS_INTERES,
                                            min_muestras=10)
print(f'\nGlosas unicas: {len(conteo)}')
print(f'Glosas con >= 10 muestras: {len(glosas_validas)}')
print(f'\nTop 20 glosas por frecuencia:')
for g, n in conteo.most_common(20):
    print(f'  {g}: {n}')
print(f'\nUltimas 10 glosas que pasan el filtro:')
for g in glosas_validas[-10:]:
    print(f'  {g}: {conteo[g]}')

## Paso 4 — Procesar videos con MediaPipe

Por cada anotación (glosa + intervalo de tiempo) en cada .eaf:
1. Lee el segmento del video correspondiente.
2. Muestrea 30 frames uniformemente.
3. Cada frame → MediaPipe → vector de 258 features.
4. Guarda `(30, 258)` en `Drive/PUCP305_processed/<GLOSA>/<id>.npy`.

**Es resumible**: salta los .npy ya existentes. Tarda 1-2 horas.

In [ ]:
from src.preprocesamiento.extraer_dataset_pucp305 import procesar_dataset_pucp305

# Opcional: limitar muestras por clase para balancear (None = sin tope)
MAX_POR_CLASE = 60

guardados = procesar_dataset_pucp305(
    LOCAL_EXTRACTED,
    salida=PROCESSED_DIR_DRIVE,
    tiers=TIERS_INTERES,
    glosas_permitidas=glosas_validas,
    max_muestras_por_clase=MAX_POR_CLASE,
)

print(f'\nResumen final: {len(guardados)} clases procesadas')
print(f'Total .npy guardados: {sum(guardados.values())}')

## Paso 5 — Cargar dataset .npy y registrar clases

Como las clases de PUCP-305 son dinámicas, las descubrimos del directorio procesado y las inyectamos en `config.CLASSES`.

In [ ]:
import numpy as np
import config

# Descubrir clases finales del directorio en Drive
clases_finales = sorted(
    d.name for d in PROCESSED_DIR_DRIVE.iterdir()
    if d.is_dir() and any(d.glob('*.npy'))
)
config.set_classes(clases_finales)
print(f'Clases activas: {len(config.CLASSES)}')
print(f'Primeras 10: {config.CLASSES[:10]}')

# Cargar todos los .npy
X, y = [], []
for idx, clase in enumerate(config.CLASSES):
    for archivo in sorted((PROCESSED_DIR_DRIVE / clase).glob('*.npy')):
        X.append(np.load(archivo))
        y.append(idx)

X = np.array(X, dtype=np.float32)
y = np.array(y)
print(f'\nDataset: X={X.shape}, y={y.shape}')
print(f'Muestras por clase: min={np.bincount(y).min()}, '
      f'max={np.bincount(y).max()}, media={np.bincount(y).mean():.1f}')

In [ ]:
from sklearn.model_selection import train_test_split

X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.1, stratify=y, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=1/9, stratify=y_tmp, random_state=42)
print(f'Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}')

## Paso 6 — Entrenar modelo LSTM con augmentation

Reutiliza el código de `src/entrenamiento/entrenar.py` (Bidirectional LSTM + class weights + Dropout 0.4) y `src/utils/augmentacion.py` (mirror, ruido, escala, etc.).

In [ ]:
from src.utils.augmentacion import expandir_train_set
from src.entrenamiento.entrenar import entrenar

X_train_aug, y_train_aug = expandir_train_set(X_train, y_train, factor=4)
print(f'Train original:   {X_train.shape}')
print(f'Train aumentado:  {X_train_aug.shape}')

model, historial = entrenar(X_train_aug, y_train_aug, X_val, y_val)

## Paso 7 — Evaluación

In [ ]:
import matplotlib.pyplot as plt
from src.evaluacion.evaluar import evaluar_modelo, matriz_confusion, graficar_historial

graficar_historial(historial)
plt.savefig('curvas_pucp305.png', dpi=120, bbox_inches='tight')

y_pred = evaluar_modelo(model, X_test, y_test)
matriz_confusion(y_test, y_pred, guardar='matriz_confusion_pucp305.png')

precision = float((y_pred == y_test).mean())
print(f'\nTest accuracy PUCP-305: {precision:.2%}')

## Paso 8 — Guardar modelo en Drive

In [ ]:
import shutil
import json

modelo_final = MODELS_DIR_DRIVE / 'modelo_pucp305_final.keras'
shutil.copy('modelo_lsp_final.keras', modelo_final)
shutil.copy('curvas_pucp305.png', MODELS_DIR_DRIVE / 'curvas_pucp305.png')
shutil.copy('matriz_confusion_pucp305.png',
             MODELS_DIR_DRIVE / 'matriz_confusion_pucp305.png')

# Guardar lista de clases para reuso en inferencia
with open(MODELS_DIR_DRIVE / 'classes_pucp305.json', 'w', encoding='utf-8') as f:
    json.dump(config.CLASSES, f, ensure_ascii=False, indent=2)

print(f'Artifacts guardados en: {MODELS_DIR_DRIVE}')